# 01 — Entropy & Mutual Information

**Companion to `docs/01_information_theory.md` (Phase 0).**

This notebook is **self-contained**: we build every information-theoretic
quantity from scratch with numpy, so you see exactly what the formulas compute.
The goal is to *feel* the one fact the whole subject rests on:

$$ I(X;Y\mid Z) = 0 \iff X \perp Y \mid Z $$


In [1]:
import numpy as np
rng = np.random.default_rng(0)

## 0. Build the quantities from scratch

All variables are integer-coded discrete arrays. We use the empirical
(plug-in) distribution: probabilities are just normalized counts.


In [2]:
def _counts(*cols):
    """Counts of each unique joint configuration of the given columns."""
    data = np.column_stack([np.asarray(c).reshape(-1) for c in cols])
    _, inv = np.unique(data, axis=0, return_inverse=True)
    return np.bincount(inv)

def entropy(*cols, base=None):
    """Shannon (joint) entropy H(cols) = -sum p log p."""
    counts = _counts(*cols)
    p = counts / counts.sum()
    h = -np.sum(p * np.log(p + (p == 0)))     # 0 log 0 := 0
    return h / np.log(base) if base else float(h)

def conditional_entropy(x, *given, base=None):
    """H(X | given) = H(X, given) - H(given)."""
    return entropy(x, *given, base=base) - entropy(*given, base=base)

def mutual_information(x, y, base=None):
    """I(X;Y) = H(X) + H(Y) - H(X,Y)."""
    return max(0.0, entropy(x, base=base) + entropy(y, base=base) - entropy(x, y, base=base))

def conditional_mutual_information(x, y, z=None, base=None):
    """I(X;Y|Z) = H(X,Z) + H(Y,Z) - H(X,Y,Z) - H(Z)."""
    if z is None:
        return mutual_information(x, y, base=base)
    zs = list(z) if isinstance(z, (list, tuple)) else [z]
    return max(0.0, entropy(x, *zs, base=base) + entropy(y, *zs, base=base)
                    - entropy(x, y, *zs, base=base) - entropy(*zs, base=base))

## 1. Entropy: uncertainty in one variable

A fair coin has 1 bit of entropy; a biased coin has less; a constant has zero.


In [3]:
fair   = rng.integers(0, 2, 10000)
biased = (rng.random(10000) < 0.9).astype(int)
const  = np.zeros(10000, dtype=int)

for name, x in [("fair coin", fair), ("biased (p=0.9)", biased), ("constant", const)]:
    print(f"{name:>16}: H = {entropy(x, base=2):.4f} bits")

       fair coin: H = 1.0000 bits
  biased (p=0.9): H = 0.4684 bits
        constant: H = -0.0000 bits


## 2. Mutual information: shared information

`I(X;Y)` is how much knowing one reduces uncertainty about the other.
For identical variables it equals the entropy; for independent ones it's ~0.


In [4]:
x = rng.integers(0, 3, 10000)
y_same = x.copy()
y_indep = rng.integers(0, 3, 10000)

print(f"I(X;X)       = {mutual_information(x, y_same):.4f}  (== H(X) = {entropy(x):.4f})")
print(f"I(X;Y_indep) = {mutual_information(x, y_indep):.4f}  (~0)")

I(X;X)       = 1.0986  (== H(X) = 1.0986)
I(X;Y_indep) = 0.0002  (~0)


## 3. The identity  I(X;Y) = H(X) − H(X|Y)

Two routes to the same number — verify they agree.


In [5]:
a = rng.integers(0, 2, 10000)
b = (a ^ (rng.random(10000) < 0.2)).astype(int)     # noisy copy of a

lhs = mutual_information(a, b)
rhs = entropy(a) - conditional_entropy(a, b)
print(f"I(A;B)        = {lhs:.5f}")
print(f"H(A) - H(A|B) = {rhs:.5f}")
print("match:", np.isclose(lhs, rhs))

I(A;B)        = 0.18900
H(A) - H(A|B) = 0.18900
match: True


## 4. Conditional mutual information — the workhorse

Build a chain `A → B → C`. Then `A` and `C` are dependent, but become
**independent once you condition on `B`** — the middle variable blocks the path.
That is `I(A;C|B) ≈ 0`, the signature of conditional independence.


In [6]:
A = rng.integers(0, 2, 20000)
B = (A ^ (rng.random(20000) < 0.15)).astype(int)
C = (B ^ (rng.random(20000) < 0.15)).astype(int)

print(f"I(A;C)     = {mutual_information(A, C):.4f}   -> dependent")
print(f"I(A;C | B) = {conditional_mutual_information(A, C, B):.4f}   -> ~0: B blocks the path")

I(A;C)     = 0.1228   -> dependent


I(A;C | B) = 0.0000   -> ~0: B blocks the path


## 5. The collider does the opposite

For a collider `A → C ← B`, parents `A` and `B` are **marginally independent**
but **become dependent once you condition on the child `C`**. This is precisely
why a Markov Blanket must include *spouses* (notebook 03).


In [7]:
A = rng.integers(0, 2, 20000)
B = rng.integers(0, 2, 20000)
logit = -1.5 + 1.6*A + 1.6*B
C = (rng.random(20000) < 1/(1+np.exp(-logit))).astype(int)

print(f"I(A;B)     = {mutual_information(A, B):.4f}   -> ~0: parents independent")
print(f"I(A;B | C) = {conditional_mutual_information(A, B, C):.4f}   -> > 0: conditioning on collider opens the path")

I(A;B)     = 0.0001   -> ~0: parents independent


I(A;B | C) = 0.0061   -> > 0: conditioning on collider opens the path


## 6. Small-sample bias (why estimation is hard)

The plug-in estimator is **biased upward**: truly independent variables show
positive MI at small N, shrinking toward 0 as N grows. This is the core reason
CI tests get unreliable with large conditioning sets — each stratum holds few
samples.


In [8]:
for n in [50, 200, 1000, 5000, 50000]:
    xs = rng.integers(0, 4, n); ys = rng.integers(0, 4, n)   # independent
    print(f"N={n:>6}:  I(X;Y) estimate = {mutual_information(xs, ys):.4f}  (true = 0)")

N=    50:  I(X;Y) estimate = 0.3147  (true = 0)
N=   200:  I(X;Y) estimate = 0.0184  (true = 0)
N=  1000:  I(X;Y) estimate = 0.0040  (true = 0)


N=  5000:  I(X;Y) estimate = 0.0006  (true = 0)


N= 50000:  I(X;Y) estimate = 0.0000  (true = 0)


### Takeaways

- `H` = uncertainty; `I(X;Y)` = shared information; both ≥ 0.
- `I(X;Y|Z) = 0 ⟺ X ⊥ Y | Z` — the definition we will *test* in notebook 02.
- Chains/forks are **blocked** by conditioning; colliders are **opened** by it.
- MI estimates are biased upward at small N → keep conditioning sets small.

**Exercises:** `exercises/01_information_theory.md`.
